## Signal 1: Noisy Sine Wave from a Cheap Oscillator

We simulate a real‑world analog sine wave with typical imperfections:
- **DC bias** – a small constant offset often present in cheap circuits.
- **Frequency jitter** – tiny random frequency fluctuations, causing phase drift (characteristic of unstable oscillators).
- **Additive Gaussian noise** – thermal/sensor noise.

**Parameters** (chosen to represent a low‑quality 5 Hz oscillator sampled at 1 kHz over 2 seconds):
- Sampling rate `fs = 1000` Hz
- Nominal frequency `f_nom = 5` Hz
- Amplitude `1.0`
- DC offset `0.2`
- Noise std `0.15`
- Frequency jitter std `0.1` Hz (introduces visible phase wander)

We plot both the pure reference sine and the final “measured” signal, and export the **measured signal only** to `sine_measured.csv`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- Parameters (tunable) ----------
fs = 1000                 # Sampling rate [Hz]
duration = 2.0            # Signal length [s]
f_nom = 5.0               # Nominal frequency [Hz]
amplitude = 1.0
dc_offset = 0.2           # DC bias
noise_std = 0.15          # Gaussian noise standard deviation
freq_jitter_std = 0.1     # Std of random frequency fluctuations [Hz]

# ---------- Time vector ----------
t = np.arange(0, duration, 1/fs)   # 0 to duration with step 1/fs
N = len(t)
dt = 1/fs

# ---------- Generate ideal (clean) sine ----------
ideal_sine = amplitude * np.sin(2 * np.pi * f_nom * t)

# ---------- Add frequency jitter (phase noise) ----------
# Random frequency deviation at each sample
freq_deviation = np.random.normal(0, freq_jitter_std, size=N)
# Phase jitter = 2π * cumulative sum of (frequency_deviation * dt)
phase_jitter = 2 * np.pi * np.cumsum(freq_deviation * dt)
# Total phase with nominal + jitter
total_phase = 2 * np.pi * f_nom * t + phase_jitter

# ---------- Build the measured signal ----------
# Start with noisy+jittery sine, add DC offset, add noise
measured_signal = amplitude * np.sin(total_phase) + dc_offset
measured_signal += np.random.normal(0, noise_std, size=N)

# ---------- Plot ----------
plt.figure(figsize=(10, 4))
plt.plot(t, ideal_sine, '--', linewidth=1.5, label='Ideal clean sine')
plt.plot(t, measured_signal, alpha=0.8, linewidth=0.8, label='Measured signal (noisy, biased, jittered)')
plt.axhline(dc_offset, color='gray', linestyle=':', label=f'DC offset = {dc_offset}')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.title('Signal 1 – Noisy Sine from a Cheap Oscillator')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------- Export measured signal to CSV (numpy) ----------
csv_filename = 'sine_measured.csv'
# Combine time and signal as columns: row[0]=time, row[1]=amplitude
data_to_save = np.column_stack((t, measured_signal))
header = 'Time (s),Amplitude'
np.savetxt(csv_filename, data_to_save, delimiter=',', header=header, comments='')
print(f'CSV exported to {csv_filename}')

## Signal 2: Laboratory Temperature Sensor with Real-World Artefacts

We simulate a temperature sensor monitoring a heated stage in a typical lab setup.
The stage warms from room temperature (25 °C) to 40 °C with an exponential rise.
Four common measurement imperfections are added, representing a realistic data‑acquisition environment:

1. **Gaussian electronic noise** – inherent sensor/amplifier noise (σ = 0.2 °C).
2. **Positive voltage spikes** – occasional sharp spikes (3–5 °C) caused by nearby relay switching or motor brushes; 0.5 % probability per sample.
3. **Slow sensor drift** – electronics warm-up causes a gentle baseline drift of 0.05 °C/s (total +0.5 °C over 10 s).
4. **50 Hz power‑line interference** – capacitive coupling from mains wiring induces a faint 50 Hz hum (amplitude 0.05 °C).

**Measurement setup**
- Sampling rate: 100 Hz
- Duration: 10 seconds

**Outputs**
- Time‑domain plot comparing the clean thermal process (dashed) with the degraded measurement.
- CSV export of the **measured signal only** (`temperature_measurement.csv`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------- Parameters ----------
fs = 100                  # Sampling rate [Hz]
duration = 10.0           # Measurement length [s]
t = np.arange(0, duration, 1/fs)
N = len(t)

# Thermal process
T_ambient = 25.0          # °C
T_target = 40.0           # °C
tau = 2.0                 # Time constant [s]
t_heat_start = 1.0        # Heater turns on at t = 1 s

# Noise
noise_std = 0.2           # Gaussian noise std [°C]

# Spikes (positive only)
spike_prob = 0.005        # per sample
spike_amp_min = 3.0       # °C
spike_amp_max = 5.0       # °C

# Drift
drift_slope = 0.05        # °C/s  (total drift = 0.5 °C over 10 s)

# 50 Hz mains hum
hum_amplitude = 0.05      # °C
hum_freq = 50.0           # Hz

# ---------- Generate clean temperature (ideal physical process) ----------
clean_temp = np.full_like(t, T_ambient)
heating_region = t >= t_heat_start
clean_temp[heating_region] = T_ambient + (T_target - T_ambient) * \
                             (1 - np.exp(-(t[heating_region] - t_heat_start) / tau))

# ---------- Artefacts ----------
# 1. Slow drift (sensor electronics warming up)
drift = drift_slope * t

# 2. 50 Hz mains hum
hum = hum_amplitude * np.sin(2 * np.pi * hum_freq * t)

# 3. Gaussian measurement noise
noise = np.random.normal(0, noise_std, size=N)

# 4. Positive spikes (random placement & random amplitude)
spike_mask = np.random.rand(N) < spike_prob
spike_amplitudes = np.random.uniform(spike_amp_min, spike_amp_max, size=N)
spikes = spike_mask * spike_amplitudes   # non-zero only where spikes occur

# ---------- Build final measured signal ----------
measured = clean_temp + drift + hum + noise + spikes

# ---------- Plot ----------
plt.figure(figsize=(10, 4.5))
plt.plot(t, clean_temp, '--', linewidth=1.5, label='Clean thermal process (ideal)')
plt.plot(t, measured, linewidth=0.9, alpha=0.9, label='Measured signal (noisy, drift, hum, spikes)')
# Mark spike positions for visibility
spike_indices = np.where(spike_mask)[0]
if len(spike_indices) > 0:
    plt.scatter(t[spike_indices], measured[spike_indices],
                color='red', s=25, zorder=5, label='Spikes')
plt.xlabel('Time [s]')
plt.ylabel('Temperature [°C]')
plt.title('Signal 2 – Temperature Sensor with Real-World Artefacts')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---------- Export measured signal to CSV ----------
csv_filename = 'temperature_measurement.csv'
data_to_save = np.column_stack((t, measured))
header = 'Time (s),Temperature (degC)'
np.savetxt(csv_filename, data_to_save, delimiter=',', header=header, comments='')
print(f'CSV exported to {csv_filename}')

## Signal 3: Electrocardiogram (ECG) Recording in a Noisy Clinical Environment

We generate a realistic single‑lead (Lead II) ECG signal at 60 bpm, then corrupt it with typical
clinical artefacts:

1. **Baseline wander** – low‑frequency drift (0.15 Hz, amplitude 0.15 mV) caused by the patient’s
   breathing and subtle electrode motion.
2. **50 Hz power‑line interference** – mains hum capacitively coupled into the body/leads
   (amplitude 0.05 mV).
3. **Muscle noise (EMG)** – high‑frequency noise mimicking skeletal muscle activity, modelled as
   broadband Gaussian noise (σ = 0.03 mV).
4. **Electrode motion artefacts** – abrupt baseline jumps when the electrode–skin contact is
   momentarily disturbed.

**ECG generation**  
A single heartbeat is constructed as the sum of Gaussians, producing a classic P–QRS–T complex:
- P wave (atrial depolarisation)
- Q, R, S waves (ventricular depolarisation)
- T wave (ventricular repolarisation)

This beat is repeated every 1 s (60 bpm) for 10 seconds.

**Outputs**
- Time‑domain plot comparing the clean ECG with the corrupted recording.
- CSV export of the corrupted ECG (`ecg_recording.csv`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ================== Parameters ==================
fs = 500                    # Sampling rate [Hz]
duration = 10.0             # Recording length [s]
t = np.arange(0, duration, 1/fs)
N = len(t)

# Heartbeat placement (10 beats, centres at 0.5 s intervals)
beat_centres = np.arange(0.5, duration, 1.0)   # 0.5, 1.5, ..., 9.5 s

# ECG beat template (Lead II, approximate amplitudes)
def ecg_beat(t_rel):
    """Return a single heartbeat at relative time t_rel (0 = R peak)."""
    p = 0.1  * np.exp(-0.5 * ((t_rel + 0.2) / 0.08)**2)   # P wave
    q = -0.1 * np.exp(-0.5 * ((t_rel + 0.05) / 0.03)**2)  # Q wave
    r = 1.0  * np.exp(-0.5 * (t_rel / 0.03)**2)           # R wave
    s = -0.3 * np.exp(-0.5 * ((t_rel - 0.05) / 0.03)**2)  # S wave
    t_w = 0.3 * np.exp(-0.5 * ((t_rel - 0.2) / 0.1)**2)   # T wave
    return p + q + r + s + t_w

# ================== Clean ECG ==================
clean_ecg = np.zeros(N)
for centre in beat_centres:
    t_rel = t - centre                          # time relative to beat centre
    # Only add beat where relative time is within ±0.4 s (avoid far tails)
    mask = np.abs(t_rel) <= 0.4
    clean_ecg[mask] += ecg_beat(t_rel[mask])

# ================== Artefacts ==================
# 1. Baseline wander (respiration, low frequency)
bw_freq = 0.15          # Hz
bw_amp = 0.15           # mV
baseline_wander = bw_amp * np.sin(2 * np.pi * bw_freq * t)

# 2. 50 Hz power-line interference
hum_amp = 0.05          # mV
hum_50hz = hum_amp * np.sin(2 * np.pi * 50 * t)

# 3. Muscle noise (broadband, approximating high‑frequency EMG)
muscle_noise_std = 0.03  # mV
muscle_noise = np.random.normal(0, muscle_noise_std, size=N)

# 4. Electrode motion artefacts (abrupt baseline jumps)
electrode_jumps = np.zeros(N)
# Define jump times (in seconds) and amplitudes (mV)
jump_times =   [2.3, 5.7, 8.1]
jump_values =  [0.3, -0.2, 0.4]
for j_time, j_val in zip(jump_times, jump_values):
    idx = np.searchsorted(t, j_time)
    electrode_jumps[idx:] += j_val          # step from that point onward

# ================== Final measured ECG ==================
measured_ecg = clean_ecg + baseline_wander + hum_50hz + muscle_noise + electrode_jumps

# ================== Plot ==================
plt.figure(figsize=(12, 4.5))
plt.plot(t, clean_ecg, '--', linewidth=1.2, label='Clean ECG (ideal)')
plt.plot(t, measured_ecg, linewidth=0.8, alpha=0.9, label='Measured ECG (with artefacts)')
# Mark electrode motion jump times
for j_time in jump_times:
    plt.axvline(j_time, color='red', linestyle=':', alpha=0.6, label='Electrode motion' if j_time == jump_times[0] else "")
plt.xlabel('Time [s]')
plt.ylabel('ECG amplitude [mV]')
plt.title('Signal 3 – ECG Recording with Baseline Wander, 50 Hz Hum, Muscle Noise & Motion Artefacts')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ================== Export measured ECG to CSV ==================
csv_filename = 'ecg_recording.csv'
data_to_save = np.column_stack((t, measured_ecg))
header = 'Time (s),ECG (mV)'
np.savetxt(csv_filename, data_to_save, delimiter=',', header=header, comments='')
print(f'CSV exported to {csv_filename}')

## Interactive Signal Generator

Use the dropdown to select a signal type, adjust its parameters, and click **Update Plot**.
You can toggle artefacts on/off and export the final noisy signal to CSV.

- **Dependencies:** `numpy`, `matplotlib`, `ipywidgets`
- If `ipywidgets` is not installed, run: `!pip install ipywidgets`

In [ ]:
import sys
!{sys.executable} -m pip install ipywidgets -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

# -------------------- Global plot area --------------------
out = widgets.Output()

# -------------------- Signal generation functions --------------------
def generate_sine(fs=1000, duration=2.0, f=5.0, amplitude=1.0, dc_offset=0.2,
                  noise_std=0.15, jitter_std=0.1):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    ideal = amplitude * np.sin(2 * np.pi * f * t)
    freq_dev = np.random.normal(0, jitter_std, size=N)
    phase_jitter = 2 * np.pi * np.cumsum(freq_dev / fs)
    total_phase = 2 * np.pi * f * t + phase_jitter
    measured = amplitude * np.sin(total_phase) + dc_offset
    measured += np.random.normal(0, noise_std, size=N)
    return t, ideal, measured

def generate_temperature(fs=100, duration=10.0, T_ambient=25.0, T_target=40.0, tau=2.0,
                         heat_start=1.0, noise_std=0.2, spike_prob=0.005,
                         spike_min=3.0, spike_max=5.0, drift_slope=0.05, hum_amp=0.05,
                         hum_freq=50.0, include_drift=True, include_hum=True, include_spikes=True):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    clean = np.full_like(t, T_ambient)
    mask = t >= heat_start
    clean[mask] = T_ambient + (T_target - T_ambient) * (1 - np.exp(-(t[mask] - heat_start) / tau))
    
    measured = clean + np.random.normal(0, noise_std, size=N)
    if include_drift:
        measured += drift_slope * t
    if include_hum:
        measured += hum_amp * np.sin(2 * np.pi * hum_freq * t)
    if include_spikes:
        spike_mask_arr = np.random.rand(N) < spike_prob
        spike_amps = np.random.uniform(spike_min, spike_max, size=N)
        measured += spike_mask_arr * spike_amps
    return t, clean, measured

def ecg_beat(t_rel):
    """Single heartbeat template (Gaussians)."""
    p = 0.1  * np.exp(-0.5 * ((t_rel + 0.2) / 0.08)**2)
    q = -0.1 * np.exp(-0.5 * ((t_rel + 0.05) / 0.03)**2)
    r = 1.0  * np.exp(-0.5 * (t_rel / 0.03)**2)
    s = -0.3 * np.exp(-0.5 * ((t_rel - 0.05) / 0.03)**2)
    t_w = 0.3 * np.exp(-0.5 * ((t_rel - 0.2) / 0.1)**2)
    return p + q + r + s + t_w

def generate_ecg(fs=500, duration=10.0, heart_rate=60.0, bw_amp=0.15, bw_freq=0.15,
                 hum_amp=0.05, hum_freq=50.0, emg_std=0.03,
                 include_bw=True, include_hum=True, include_emg=True, include_motion=True,
                 motion_times_str="2.3,5.7,8.1", motion_amps_str="0.3,-0.2,0.4",
                 motion_decay_tau=0.5):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    beat_interval = 60.0 / heart_rate
    beat_centres = np.arange(0.5, duration, beat_interval)
    
    clean = np.zeros(N)
    for centre in beat_centres:
        t_rel = t - centre
        mask = np.abs(t_rel) <= 0.4
        clean[mask] += ecg_beat(t_rel[mask])
    
    measured = clean.copy()
    if include_bw:
        measured += bw_amp * np.sin(2 * np.pi * bw_freq * t)
    if include_hum:
        measured += hum_amp * np.sin(2 * np.pi * hum_freq * t)
    if include_emg:
        measured += np.random.normal(0, emg_std, size=N)
    if include_motion:
        # Parse motion times and amplitudes
        try:
            m_times = [float(x.strip()) for x in motion_times_str.split(',')]
            m_amps  = [float(x.strip()) for x in motion_amps_str.split(',')]
        except:
            m_times, m_amps = [], []
        for mt, ma in zip(m_times, m_amps):
            # Create decaying step: exponential decay with time constant motion_decay_tau
            idx = np.searchsorted(t, mt)
            if idx < N:
                decay = np.exp(-(t[idx:] - t[idx]) / motion_decay_tau)
                measured[idx:] += ma * decay
    return t, clean, measured

# -------------------- UI update function --------------------
def update_plot(b):
    with out:
        clear_output(wait=True)
        signal = signal_selector.value
        if signal == 'Sine Wave (Noisy Oscillator)':
            t, clean, noisy = generate_sine(
                fs=fs_sine.value, duration=dur_sine.value, f=freq_sine.value,
                amplitude=amp_sine.value, dc_offset=dc_sine.value,
                noise_std=noise_sine.value, jitter_std=jitter_sine.value)
            title = 'Sine Wave – Noisy Oscillator'
            csv_default = 'sine_measured.csv'
        elif signal == 'Temperature Sensor':
            t, clean, noisy = generate_temperature(
                fs=fs_temp.value, duration=dur_temp.value, T_ambient=Tamb.value,
                T_target=Ttarget.value, tau=tau_temp.value, heat_start=heat_start.value,
                noise_std=noise_temp.value, spike_prob=spike_prob.value,
                spike_min=spike_min.value, spike_max=spike_max.value,
                drift_slope=drift_slope.value, hum_amp=hum_temp.value, hum_freq=hum_freq_temp.value,
                include_drift=drift_toggle.value, include_hum=hum_temp_toggle.value,
                include_spikes=spikes_toggle.value)
            title = 'Temperature Sensor with Artefacts'
            csv_default = 'temperature_measurement.csv'
        else:  # ECG
            t, clean, noisy = generate_ecg(
                fs=fs_ecg.value, duration=dur_ecg.value, heart_rate=hr_ecg.value,
                bw_amp=bw_amp.value, bw_freq=bw_freq.value,
                hum_amp=hum_ecg.value, hum_freq=hum_freq_ecg.value,
                emg_std=emg_std.value,
                include_bw=bw_toggle.value, include_hum=hum_ecg_toggle.value,
                include_emg=emg_toggle.value, include_motion=motion_toggle.value,
                motion_times_str=motion_times.value, motion_amps_str=motion_amps.value,
                motion_decay_tau=motion_decay.value)
            title = 'ECG Recording with Artefacts'
            csv_default = 'ecg_recording.csv'
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(t, clean, '--', linewidth=1.2, label='Clean (ideal)')
        ax.plot(t, noisy, linewidth=0.8, alpha=0.9, label='Measured (noisy)')
        ax.set_xlabel('Time [s]')
        ax.set_ylabel('Amplitude')
        ax.set_title(title)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Store data for export
        update_plot.current_t = t
        update_plot.current_noisy = noisy
        update_plot.current_csv_default = csv_default

# Attach attributes to function for export
update_plot.current_t = None
update_plot.current_noisy = None
update_plot.current_csv_default = 'signal.csv'

def export_csv(b):
    with out:
        if update_plot.current_noisy is None:
            print("Generate a signal first (click Update Plot).")
            return
        filename = csv_filename.value.strip()
        if not filename:
            filename = update_plot.current_csv_default
        data = np.column_stack((update_plot.current_t, update_plot.current_noisy))
        np.savetxt(filename, data, delimiter=',', header='Time (s),Amplitude', comments='')
        print(f"Exported to {filename}")

# -------------------- Signal selector --------------------
signal_selector = widgets.Dropdown(
    options=['Sine Wave (Noisy Oscillator)', 'Temperature Sensor', 'ECG'],
    value='Sine Wave (Noisy Oscillator)',
    description='Signal:',
)

# -------------------- Sine wave controls --------------------
fs_sine = widgets.IntSlider(value=1000, min=100, max=5000, step=100, description='Fs [Hz]')
dur_sine = widgets.FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description='Duration [s]')
freq_sine = widgets.FloatSlider(value=5.0, min=0.1, max=50.0, step=0.1, description='Freq [Hz]')
amp_sine = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Amplitude')
dc_sine = widgets.FloatSlider(value=0.2, min=-1.0, max=1.0, step=0.05, description='DC offset')
noise_sine = widgets.FloatSlider(value=0.15, min=0.0, max=1.0, step=0.01, description='Noise std')
jitter_sine = widgets.FloatSlider(value=0.1, min=0.0, max=1.0, step=0.01, description='Freq jitter std')
sine_box = widgets.VBox([fs_sine, dur_sine, freq_sine, amp_sine, dc_sine, noise_sine, jitter_sine])

# -------------------- Temperature sensor controls --------------------
fs_temp = widgets.IntSlider(value=100, min=10, max=1000, step=10, description='Fs [Hz]')
dur_temp = widgets.FloatSlider(value=10.0, min=2.0, max=60.0, step=1.0, description='Duration [s]')
Tamb = widgets.FloatSlider(value=25.0, min=0.0, max=50.0, step=0.5, description='Ambient [°C]')
Ttarget = widgets.FloatSlider(value=40.0, min=20.0, max=100.0, step=0.5, description='Target [°C]')
tau_temp = widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description='Time const [s]')
heat_start = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description='Heat start [s]')
noise_temp = widgets.FloatSlider(value=0.2, min=0.0, max=2.0, step=0.05, description='Noise std [°C]')
spike_prob = widgets.FloatSlider(value=0.005, min=0.0, max=0.1, step=0.001, description='Spike prob')
spike_min = widgets.FloatSlider(value=3.0, min=0.5, max=10.0, step=0.1, description='Spike min [°C]')
spike_max = widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.1, description='Spike max [°C]')
drift_slope = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='Drift [°C/s]')
hum_temp = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='Hum amp [°C]')
hum_freq_temp = widgets.FloatSlider(value=50.0, min=10.0, max=100.0, step=1.0, description='Hum freq [Hz]')
# Toggles
drift_toggle = widgets.Checkbox(value=True, description='Include drift')
hum_temp_toggle = widgets.Checkbox(value=True, description='Include 50 Hz hum')
spikes_toggle = widgets.Checkbox(value=True, description='Include spikes')
temp_box = widgets.VBox([fs_temp, dur_temp, Tamb, Ttarget, tau_temp, heat_start,
                         noise_temp, spike_prob, spike_min, spike_max,
                         drift_slope, drift_toggle, hum_temp, hum_freq_temp,
                         hum_temp_toggle, spikes_toggle])

# -------------------- ECG controls --------------------
fs_ecg = widgets.IntSlider(value=500, min=100, max=2000, step=100, description='Fs [Hz]')
dur_ecg = widgets.FloatSlider(value=10.0, min=2.0, max=30.0, step=1.0, description='Duration [s]')
hr_ecg = widgets.FloatSlider(value=60.0, min=30.0, max=150.0, step=1.0, description='Heart rate [bpm]')
bw_amp = widgets.FloatSlider(value=0.15, min=0.0, max=1.0, step=0.01, description='BW amplitude')
bw_freq = widgets.FloatSlider(value=0.15, min=0.05, max=2.0, step=0.01, description='BW freq [Hz]')
hum_ecg = widgets.FloatSlider(value=0.05, min=0.0, max=0.5, step=0.01, description='Hum amplitude')
hum_freq_ecg = widgets.FloatSlider(value=50.0, min=10.0, max=100.0, step=1.0, description='Hum freq [Hz]')
emg_std = widgets.FloatSlider(value=0.03, min=0.0, max=0.3, step=0.005, description='Muscle noise std')
motion_times = widgets.Text(value="2.3,5.7,8.1", description='Motion times [s]', layout=widgets.Layout(width='300px'))
motion_amps = widgets.Text(value="0.3,-0.2,0.4", description='Motion amplitudes', layout=widgets.Layout(width='300px'))
motion_decay = widgets.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description='Motion decay tau')
# Toggles
bw_toggle = widgets.Checkbox(value=True, description='Baseline wander')
hum_ecg_toggle = widgets.Checkbox(value=True, description='50 Hz hum')
emg_toggle = widgets.Checkbox(value=True, description='Muscle noise')
motion_toggle = widgets.Checkbox(value=True, description='Electrode motion')
ecg_box = widgets.VBox([fs_ecg, dur_ecg, hr_ecg, bw_amp, bw_freq, bw_toggle,
                        hum_ecg, hum_freq_ecg, hum_ecg_toggle,
                        emg_std, emg_toggle,
                        motion_times, motion_amps, motion_decay, motion_toggle])

# -------------------- Visibility handler --------------------
controls_dict = {
    'Sine Wave (Noisy Oscillator)': sine_box,
    'Temperature Sensor': temp_box,
    'ECG': ecg_box
}
controls_area = widgets.VBox([sine_box])  # placeholder

def on_signal_change(change):
    controls_area.children = [controls_dict[change['new']]]
signal_selector.observe(on_signal_change, names='value')
# initialise
on_signal_change({'new': signal_selector.value})

# -------------------- Buttons and export --------------------
update_btn = widgets.Button(description='Update Plot', button_style='success')
export_btn = widgets.Button(description='Export CSV', button_style='info')
csv_filename = widgets.Text(value='', placeholder='filename.csv', description='File:')

update_btn.on_click(update_plot)
export_btn.on_click(export_csv)

# -------------------- Layout --------------------
ui = widgets.VBox([
    signal_selector,
    controls_area,
    widgets.HBox([update_btn, export_btn, csv_filename]),
    out
])

display(ui)

## Interactive Signal Generator

Select a signal type, adjust its parameters, and click **Generate** to view the result.
You can toggle artefacts on/off and export the noisy signal to CSV.

**Dependencies:** `numpy`, `matplotlib`, `ipywidgets` (install with `!pip install ipywidgets` if missing).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------- Signal generation functions (unchanged) --------------------
def generate_sine(fs=1000, duration=2.0, f=5.0, amplitude=1.0, dc_offset=0.2,
                  noise_std=0.15, jitter_std=0.1):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    ideal = amplitude * np.sin(2 * np.pi * f * t)
    freq_dev = np.random.normal(0, jitter_std, size=N)
    phase_jitter = 2 * np.pi * np.cumsum(freq_dev / fs)
    total_phase = 2 * np.pi * f * t + phase_jitter
    measured = amplitude * np.sin(total_phase) + dc_offset + np.random.normal(0, noise_std, size=N)
    return t, ideal, measured

def generate_temperature(fs=100, duration=10.0, T_ambient=25.0, T_target=40.0, tau=2.0,
                         heat_start=1.0, noise_std=0.2, spike_prob=0.005,
                         spike_min=3.0, spike_max=5.0, drift_slope=0.05, hum_amp=0.05,
                         hum_freq=50.0, include_drift=True, include_hum=True, 
                         include_spikes=True, include_noise=True):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    clean = np.full_like(t, T_ambient)
    mask = t >= heat_start
    clean[mask] = T_ambient + (T_target - T_ambient) * (1 - np.exp(-(t[mask] - heat_start) / tau))
    
    measured = clean.copy()
    if include_noise:
        measured += np.random.normal(0, noise_std, size=N)
    if include_drift:
        measured += drift_slope * t
    if include_hum:
        measured += hum_amp * np.sin(2 * np.pi * hum_freq * t)
    if include_spikes:
        spike_mask_arr = np.random.rand(N) < spike_prob
        spike_amps = np.random.uniform(spike_min, spike_max, size=N)
        measured += spike_mask_arr * spike_amps
    return t, clean, measured

def ecg_beat(t_rel):
    p = 0.1  * np.exp(-0.5 * ((t_rel + 0.2) / 0.08)**2)
    q = -0.1 * np.exp(-0.5 * ((t_rel + 0.05) / 0.03)**2)
    r = 1.0  * np.exp(-0.5 * (t_rel / 0.03)**2)
    s = -0.3 * np.exp(-0.5 * ((t_rel - 0.05) / 0.03)**2)
    t_w = 0.3 * np.exp(-0.5 * ((t_rel - 0.2) / 0.1)**2)
    return p + q + r + s + t_w

def generate_ecg(fs=500, duration=10.0, heart_rate=60.0, bw_amp=0.15, bw_freq=0.15,
                 hum_amp=0.05, hum_freq=50.0, emg_std=0.03,
                 include_bw=True, include_hum=True, include_emg=True, include_motion=True,
                 motion_times_str="2.3,5.7,8.1", motion_amps_str="0.3,-0.2,0.4",
                 motion_decay_tau=0.5):
    t = np.arange(0, duration, 1/fs)
    N = len(t)
    beat_interval = 60.0 / heart_rate
    beat_centres = np.arange(0.5, duration, beat_interval)
    clean = np.zeros(N)
    for centre in beat_centres:
        t_rel = t - centre
        mask = np.abs(t_rel) <= 0.4
        clean[mask] += ecg_beat(t_rel[mask])
    measured = clean.copy()
    if include_bw:
        measured += bw_amp * np.sin(2 * np.pi * bw_freq * t)
    if include_hum:
        measured += hum_amp * np.sin(2 * np.pi * hum_freq * t)
    if include_emg:
        measured += np.random.normal(0, emg_std, size=N)
    if include_motion:
        try:
            m_times = [float(x.strip()) for x in motion_times_str.split(',')]
            m_amps  = [float(x.strip()) for x in motion_amps_str.split(',')]
        except:
            m_times, m_amps = [], []
        for mt, ma in zip(m_times, m_amps):
            idx = np.searchsorted(t, mt)
            if idx < N:
                decay = np.exp(-(t[idx:] - t[idx]) / motion_decay_tau)
                measured[idx:] += ma * decay
    return t, clean, measured

# -------------------- Styling helper --------------------
def styled_box(children, title=None):
    items = []
    if title:
        items.append(widgets.HTML(f"<b style='font-size:14px; color:#2c3e50;'>{title}</b>"))
    items += children
    return widgets.VBox(items, layout=widgets.Layout(
        border='1px solid #bdc3c7', padding='10px', margin='5px 0px 5px 0px',
        border_radius='8px', background_color='#fafafa'
    ))

# -------------------- Create all widgets --------------------
# Header
header = widgets.HTML("<h2 style='color:#2c3e50; margin-bottom:0;'>📊 Interactive Signal Generator</h2>")
subtitle = widgets.HTML("<p style='color:#7f8c8d; margin-top:0;'>Configure parameters, generate signal, and export to CSV.</p>")

# Signal selector
signal_selector = widgets.Dropdown(
    options=['Sine Wave (Noisy Oscillator)', 'Temperature Sensor', 'ECG Recording'],
    value='Sine Wave (Noisy Oscillator)',
    description='Signal Type:',
    layout=widgets.Layout(width='50%', margin='0 0 10px 0'),
    style={'description_width': 'initial'}
)

# ---------- Sine widgets ----------
sine_fs = widgets.IntSlider(value=1000, min=100, max=5000, step=100, description='Sampling rate [Hz]', style={'description_width': 'initial'})
sine_dur = widgets.FloatSlider(value=2.0, min=0.5, max=10, step=0.5, description='Duration [s]', style={'description_width': 'initial'})
sine_freq = widgets.FloatSlider(value=5.0, min=0.1, max=50, step=0.1, description='Frequency [Hz]', style={'description_width': 'initial'})
sine_amp = widgets.FloatSlider(value=1.0, min=0.1, max=5, step=0.1, description='Amplitude', style={'description_width': 'initial'})
sine_dc = widgets.FloatSlider(value=0.2, min=-1.0, max=1.0, step=0.05, description='DC offset', style={'description_width': 'initial'})
sine_noise = widgets.FloatSlider(value=0.15, min=0, max=1.0, step=0.01, description='Noise std', style={'description_width': 'initial'})
sine_jitter = widgets.FloatSlider(value=0.1, min=0, max=1.0, step=0.01, description='Freq jitter std', style={'description_width': 'initial'})

sine_box = styled_box([sine_fs, sine_dur, sine_freq, sine_amp, sine_dc, sine_noise, sine_jitter],
                      title="Sine Wave Parameters")

# ---------- Temperature widgets ----------
temp_fs = widgets.IntSlider(value=100, min=10, max=1000, step=10, description='Sampling rate [Hz]', style={'description_width': 'initial'})
temp_dur = widgets.FloatSlider(value=10.0, min=2, max=60, step=1, description='Duration [s]', style={'description_width': 'initial'})
temp_Tamb = widgets.FloatSlider(value=25.0, min=0, max=50, step=0.5, description='Ambient [°C]', style={'description_width': 'initial'})
temp_Ttarget = widgets.FloatSlider(value=40.0, min=20, max=100, step=0.5, description='Target [°C]', style={'description_width': 'initial'})
temp_tau = widgets.FloatSlider(value=2.0, min=0.1, max=10, step=0.1, description='Time constant [s]', style={'description_width': 'initial'})
temp_heat_start = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.1, description='Heat start [s]', style={'description_width': 'initial'})
temp_noise = widgets.FloatSlider(value=0.2, min=0, max=2.0, step=0.05, description='Noise std [°C]', style={'description_width': 'initial'})
temp_spike_prob = widgets.FloatSlider(value=0.005, min=0, max=0.1, step=0.001, description='Spike probability', style={'description_width': 'initial'})
temp_spike_min = widgets.FloatSlider(value=3.0, min=0.5, max=10, step=0.1, description='Spike min [°C]', style={'description_width': 'initial'})
temp_spike_max = widgets.FloatSlider(value=5.0, min=1, max=15, step=0.1, description='Spike max [°C]', style={'description_width': 'initial'})
temp_drift = widgets.FloatSlider(value=0.05, min=0, max=0.5, step=0.01, description='Drift [°C/s]', style={'description_width': 'initial'})
temp_hum_amp = widgets.FloatSlider(value=0.05, min=0, max=0.5, step=0.01, description='Hum amplitude [°C]', style={'description_width': 'initial'})
temp_hum_freq = widgets.FloatSlider(value=50.0, min=10, max=100, step=1, description='Hum frequency [Hz]', style={'description_width': 'initial'})

temp_drift_toggle = widgets.Checkbox(value=True, description='Include drift')
temp_hum_toggle = widgets.Checkbox(value=True, description='Include 50 Hz hum')
temp_spikes_toggle = widgets.Checkbox(value=True, description='Include spikes')
temp_noise_toggle = widgets.Checkbox(value=True, description='Include white noise')

temp_params_box = styled_box([temp_fs, temp_dur, temp_Tamb, temp_Ttarget, temp_tau,
                              temp_heat_start, temp_noise, temp_spike_prob, temp_spike_min,
                              temp_spike_max, temp_drift, temp_hum_amp, temp_hum_freq],
                             title="Temperature Sensor Parameters")
temp_toggles_box = styled_box([temp_drift_toggle, temp_hum_toggle, temp_spikes_toggle, temp_noise_toggle], title="Artefacts")
temp_box = widgets.VBox([temp_params_box, temp_toggles_box])

# ---------- ECG widgets ----------
ecg_fs = widgets.IntSlider(value=500, min=100, max=2000, step=100, description='Sampling rate [Hz]', style={'description_width': 'initial'})
ecg_dur = widgets.FloatSlider(value=10.0, min=2, max=30, step=1, description='Duration [s]', style={'description_width': 'initial'})
ecg_hr = widgets.FloatSlider(value=60.0, min=30, max=150, step=1, description='Heart rate [bpm]', style={'description_width': 'initial'})
ecg_bw_amp = widgets.FloatSlider(value=0.15, min=0, max=1.0, step=0.01, description='BW amplitude', style={'description_width': 'initial'})
ecg_bw_freq = widgets.FloatSlider(value=0.15, min=0.05, max=2.0, step=0.01, description='BW frequency [Hz]', style={'description_width': 'initial'})
ecg_hum_amp = widgets.FloatSlider(value=0.05, min=0, max=0.5, step=0.01, description='Hum amplitude', style={'description_width': 'initial'})
ecg_hum_freq = widgets.FloatSlider(value=50.0, min=10, max=100, step=1, description='Hum frequency [Hz]', style={'description_width': 'initial'})
ecg_emg = widgets.FloatSlider(value=0.03, min=0, max=0.3, step=0.005, description='Muscle noise std', style={'description_width': 'initial'})
ecg_motion_decay = widgets.FloatSlider(value=0.5, min=0.1, max=3.0, step=0.1, description='Motion decay tau', style={'description_width': 'initial'})
ecg_motion_times = widgets.Text(value="2.3,5.7,8.1", description='Motion times [s]', layout=widgets.Layout(width='400px'), style={'description_width': 'initial'})
ecg_motion_amps = widgets.Text(value="0.3,-0.2,0.4", description='Motion amplitudes [mV]', layout=widgets.Layout(width='400px'), style={'description_width': 'initial'})

ecg_bw_toggle = widgets.Checkbox(value=True, description='Baseline wander')
ecg_hum_toggle = widgets.Checkbox(value=True, description='50 Hz hum')
ecg_emg_toggle = widgets.Checkbox(value=True, description='Muscle noise (EMG)')
ecg_motion_toggle = widgets.Checkbox(value=True, description='Electrode motion')

ecg_params_box = styled_box([ecg_fs, ecg_dur, ecg_hr, ecg_bw_amp, ecg_bw_freq,
                             ecg_hum_amp, ecg_hum_freq, ecg_emg, ecg_motion_decay],
                            title="ECG Parameters")
ecg_motion_box = styled_box([ecg_motion_times, ecg_motion_amps], title="Electrode Motion Details")
ecg_toggles_box = styled_box([ecg_bw_toggle, ecg_hum_toggle, ecg_emg_toggle, ecg_motion_toggle], title="Artefacts")
ecg_box = widgets.VBox([ecg_params_box, ecg_motion_box, ecg_toggles_box])

# Controls container (switched by signal type)
controls_area = widgets.VBox([sine_box])

def on_signal_change(change):
    signal = change['new']
    if signal == 'Sine Wave (Noisy Oscillator)':
        controls_area.children = [sine_box]
    elif signal == 'Temperature Sensor':
        controls_area.children = [temp_box]
    else:  # ECG
        controls_area.children = [ecg_box]
signal_selector.observe(on_signal_change, names='value')

# Output plot area
out = widgets.Output()

# Buttons
generate_btn = widgets.Button(description='Generate Signal', button_style='primary',
                              layout=widgets.Layout(width='150px', margin='5px 10px 5px 0'))
export_btn = widgets.Button(description='Export CSV', button_style='info',
                            layout=widgets.Layout(width='120px', margin='5px 10px 5px 0'))
reset_btn = widgets.Button(description='Reset Defaults', button_style='warning',
                           layout=widgets.Layout(width='140px', margin='5px 0'))
csv_filename = widgets.Text(value='signal.csv', placeholder='filename.csv', description='File:',
                            layout=widgets.Layout(width='300px', margin='5px 0'),
                            style={'description_width': 'initial'})

# Store last generated data
generate_btn.current_data = (None, None)

# -------------------- Callbacks --------------------
def generate_callback(b):
    signal = signal_selector.value
    if signal == 'Sine Wave (Noisy Oscillator)':
        t, clean, noisy = generate_sine(
            fs=sine_fs.value, duration=sine_dur.value, f=sine_freq.value,
            amplitude=sine_amp.value, dc_offset=sine_dc.value,
            noise_std=sine_noise.value, jitter_std=sine_jitter.value)
        title = 'Sine Wave – Noisy Oscillator'
    elif signal == 'Temperature Sensor':
        t, clean, noisy = generate_temperature(
            fs=temp_fs.value, duration=temp_dur.value, T_ambient=temp_Tamb.value,
            T_target=temp_Ttarget.value, tau=temp_tau.value, heat_start=temp_heat_start.value,
            noise_std=temp_noise.value, spike_prob=temp_spike_prob.value,
            spike_min=temp_spike_min.value, spike_max=temp_spike_max.value,
            drift_slope=temp_drift.value, hum_amp=temp_hum_amp.value,
            hum_freq=temp_hum_freq.value,
            include_drift=temp_drift_toggle.value, include_hum=temp_hum_toggle.value,
            include_spikes=temp_spikes_toggle.value,
            include_noise=temp_noise_toggle.value)
        title = 'Temperature Sensor with Artefacts'
    else:  # ECG
        t, clean, noisy = generate_ecg(
            fs=ecg_fs.value, duration=ecg_dur.value, heart_rate=ecg_hr.value,
            bw_amp=ecg_bw_amp.value, bw_freq=ecg_bw_freq.value,
            hum_amp=ecg_hum_amp.value, hum_freq=ecg_hum_freq.value,
            emg_std=ecg_emg.value,
            include_bw=ecg_bw_toggle.value, include_hum=ecg_hum_toggle.value,
            include_emg=ecg_emg_toggle.value, include_motion=ecg_motion_toggle.value,
            motion_times_str=ecg_motion_times.value, motion_amps_str=ecg_motion_amps.value,
            motion_decay_tau=ecg_motion_decay.value)
        title = 'ECG Recording with Artefacts'

    generate_btn.current_data = (t, noisy)

    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(t, clean, '--', linewidth=1.2, label='Clean (ideal)')
        ax.plot(t, noisy, linewidth=0.8, alpha=0.9, label='Measured (noisy)')
        ax.set_xlabel('Time [s]')
        ax.set_ylabel('Amplitude')
        ax.set_title(title)
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

def export_callback(b):
    t, noisy = generate_btn.current_data
    if t is None:
        with out:
            print("Please generate a signal first.")
        return
    fname = csv_filename.value.strip()
    if not fname:
        fname = 'signal.csv'
    np.savetxt(fname, np.column_stack((t, noisy)), delimiter=',',
               header='Time (s),Amplitude', comments='')
    with out:
        print(f"Data exported to {fname}")

def reset_callback(b):
    signal = signal_selector.value
    if signal == 'Sine Wave (Noisy Oscillator)':
        sine_fs.value = 1000; sine_dur.value = 2.0; sine_freq.value = 5.0; sine_amp.value = 1.0
        sine_dc.value = 0.2; sine_noise.value = 0.15; sine_jitter.value = 0.1
    elif signal == 'Temperature Sensor':
        temp_fs.value = 100; temp_dur.value = 10.0; temp_Tamb.value = 25.0; temp_Ttarget.value = 40.0
        temp_tau.value = 2.0; temp_heat_start.value = 1.0; temp_noise.value = 0.2
        temp_spike_prob.value = 0.005; temp_spike_min.value = 3.0; temp_spike_max.value = 5.0
        temp_drift.value = 0.05; temp_hum_amp.value = 0.05; temp_hum_freq.value = 50.0
        temp_drift_toggle.value = True; temp_hum_toggle.value = True; temp_spikes_toggle.value = True; temp_noise_toggle.value = True
    else:  # ECG
        ecg_fs.value = 500; ecg_dur.value = 10.0; ecg_hr.value = 60.0; ecg_bw_amp.value = 0.15
        ecg_bw_freq.value = 0.15; ecg_hum_amp.value = 0.05; ecg_hum_freq.value = 50.0
        ecg_emg.value = 0.03; ecg_motion_decay.value = 0.5
        ecg_motion_times.value = "2.3,5.7,8.1"; ecg_motion_amps.value = "0.3,-0.2,0.4"
        ecg_bw_toggle.value = True; ecg_hum_toggle.value = True; ecg_emg_toggle.value = True
        ecg_motion_toggle.value = True
    generate_callback(None)

generate_btn.on_click(generate_callback)
export_btn.on_click(export_callback)
reset_btn.on_click(reset_callback)

# -------------------- Layout assembly --------------------
button_row = widgets.HBox([generate_btn, export_btn, reset_btn, csv_filename],
                          layout=widgets.Layout(align_items='center'))

main_ui = widgets.VBox([
    header,
    subtitle,
    signal_selector,
    controls_area,
    button_row,
    out
], layout=widgets.Layout(padding='10px', border='1px solid #dcdcdc', border_radius='10px',
                         background_color='white'))

display(main_ui)